# PA005 - High Value Customer Identification (Insiders)

# SOLUTION PLANNING

"All in one place" is an online retail store that sells second-hand products from various brands at lower prices.

With just over a year of operation, the marketing team noticed that some customers from their base purchase more expensive products more frequently, contributing significantly to the company's revenue.

Based on this insight, the marketing team decided to launch a loyalty program for the top customers in their base, named Insiders. However, the marketing team lacks the necessary knowledge to select the participating customers for this program.

As a result, this task was assigned to the company's data team, where a solution that ultimately provides a list of customers to be invited to participate in the Insiders program should be developed. Additionally, a report answering the following questions should be delivered:

1. Who are the eligible individuals to participate in the program?
    - Revenue:
        - High average ticket value
        - High customer lifetime value (LTV)
        - Low recency
        - Low churn probability
        - High LTV prediction
        - High purchase propensity
        - Cost:

        - Low return rate
        - Purchase Experience:

        - High average ratings  

2. How many customers will be part of the group?
        - Total number of customers
        - % of the insiders group
    
3.  What are the main characteristics of these customers?
    - Customer characteristics:
        - Age
        - Location
       
    - Consumption characteristics:
        - Attributes of clustering

4. What percentage of the revenue comes from the selected group?
    - Total annual revenue
    - Revenue from the Insiders group

5. What is the revenue expectation for this group in the upcoming months?
    - LTV (Customer Lifetime Value) of the Insiders group
    - Cohort analysis
    
6. What are the eligibility criteria for someone to join the group?
    - Define the frequency (1 month, 2 months, etc.).
    - The person needs to be similar to another person in the group.
    - What are the criteria for someone to be removed from the group?

7. Define the frequency (1 month, 2 months, etc.).
    - The person needs to be dissimilar to another person in the group.
    - How can we ensure that the group performs better than the rest of the customer base?

8. A/B testing.
    - Bayesian A/B testing.
    - Hypothesis testing.
    - What actions can the marketing team take to increase revenue?

9. Discounts.
    - Purchase preferences.
    - Company visits.

**SOLUTION PLANNING**

1 - Input:
    Business Problem: Hight value customers identification and selection in order to join a loyalty program.
    Dataset: sales datas from an e-commerce, collected trough an one year period.

2 - Output:
    List containing the customer identification and if they are eligible or not to the loyalty program.
    Report, answering the business questions and explaining how the selection was made.
    
3 - Tasks
    - Who are the eligible individuals to participate in the program?
        Understanding of what elegible means and what are the most valuable customer.
        
        
    - How many customers will be part of the group?
        
        
    - What are the main characteristics of these customers?
        
    
    - What percentage of the revenue comes from the selected group?
        
    
    - What is the revenue expectation for this group in the upcoming months?
        
    
    - What are the conditions for someone to be eligible for the group?
        
    
    - What are the conditions for someone to be removed from the group?
        
    
    - What assurance is there that the group is better than the rest of the base?
        
    
    - What actions can the marketing team take to increase revenue?
        


# 0 - IMPORTS </font>

In [ ]:
# Data Maniputalion and Data Analysis
import re
import warnings

import pandas                as pd
import seaborn               as sns
import numpy                 as np
import plotly.express        as px
from ydata_profiling         import ProfileReport
from matplotlib              import pyplot          as plt
import matplotlib.cm         as cm
import matplotlib            as mpl

# ML Algorithms
import umap.umap_            as umap

from sklearn.manifold        import TSNE
from sklearn                 import mixture as mx
from sklearn                 import preprocessing as pp
from sklearn                 import ensemble      as en
from sklearn                 import decomposition as dd
from sklearn                 import cluster       as cc
from sklearn                 import metrics       as mt
from sklearn                 import ensemble      as en
from yellowbrick.cluster     import KElbowVisualizer, SilhouetteVisualizer

# Loading Images and display settings
from IPython.display         import Image, display
from IPython.core.display    import HTML

warnings.filterwarnings( 'ignore' )

## 0.1 - Helper Functions

In [ ]:
def drop_and_rename_duplicate_columns(df):
    """
    Remove colunas duplicadas resultantes de um merge e renomeia as colunas para eliminar o sufixo '_x'.
    Remove linhas referentes a invoices canceladas ou devolvidas: cujo valor para a coluna `invoice_status_y` for igual a 'True'.
    
    Parâmetros:
    - df (pd.DataFrame): DataFrame resultante de um merge com possíveis colunas duplicadas.

    Retorna:
    - pd.DataFrame: DataFrame com colunas duplicadas removidas e renomeadas.
    """

    # Verifica se a coluna 'invoice_status_y' existe e filtra onde o valor é True
    if 'invoice_cancelled_y' in df.columns:
        df = df[~df['invoice_cancelled_y'].fillna(False)]
    
    # Identifica colunas com sufixos '_x' e '_y' após o merge
    duplicate_columns = [col for col in df.columns if col.endswith('_x') or col.endswith('_y')]
    
    # Cria um mapeamento para manter apenas uma ocorrência e renomear as colunas
    cols_to_keep = {}
    for col in duplicate_columns:
        base_name = col[:-2]  # Remove o sufixo '_x' ou '_y'
        if base_name not in cols_to_keep:
            # Guarda a coluna com sufixo '_x' para manter e renomear
            cols_to_keep[base_name] = col

    # Define as colunas para remoção, mantendo apenas uma de cada par duplicado
    cols_to_drop = set(duplicate_columns) - set(cols_to_keep.values())
    df = df.drop(columns=cols_to_drop)

    # Renomeia as colunas restantes, removendo o sufixo '_x'
    df = df.rename(columns={old_name: base_name for base_name, old_name in cols_to_keep.items()})

    # Filtra as colunas, removendo invoice_cancelled
    cols = ['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date','unit_price', 'customer_id', 'country']

    return df[cols]
    
def jupyter_settings():
    %matplotlib inline
    
    plt.style.use( 'bmh' )
    plt.rcParams['figure.figsize'] = [12, 5]
    plt.rcParams['font.size'] = 11
    
    display( HTML( '<style>.container { width:100% !important; }</style>') )
    pd.options.display.max_columns = None
    pd.options.display.max_rows = None
    pd.set_option( 'display.expand_frame_repr', False )
   
    sns.set()
jupyter_settings()

### Visualization Settings
%matplotlib inline
%config InlineBackend.figure_format = 'svg'

mpl.style.use( 'ggplot' )
mpl.rcParams['axes.facecolor']      = 'white'
mpl.rcParams['axes.linewidth']      = 1
mpl.rcParams['xtick.color']         = 'black'
mpl.rcParams['ytick.color']         = 'black'
mpl.rcParams['grid.color']          = 'lightgray'
mpl.rcParams['figure.dpi']          = 120
mpl.rcParams['axes.grid']           = True
mpl.rcParams['font.size']           = 10
mpl.rcParams['figure.figsize']      = [12, 5]
mpl.rcParams['font.size']           = 11

# Palette Setting
color_palette = ['#82B1FF', '#FF9100', '#40C4FF', '#A7FFEB', '#CCFF90', '#616161']

# Setting as the palette
sns.set_palette(sns.color_palette(color_palette))

# Display
sns.color_palette(color_palette)

# 1 - DATA LOADING

## 1.1 - LOADING DATA

In [ ]:
df = pd.read_csv("../data/raw/Ecommerce.csv")

# drop extra column
df = df.drop( columns=['Unnamed: 8'], axis=1)

## 1.2 - DATA DESCRIPTIVE

### 1.2.1 - RENAME COLUMNS

In [ ]:
df1 = df.copy()

In [ ]:
cols_new = ['invoice_no','stock_code','description','quantity','invoice_date','unit_price','customer_id','country']
df1.columns = cols_new

### 1.2.2 - DATA DIMENSIONS

In [ ]:
print('Number of columns: {}'.format( df1.shape[1] ) )
print('Number of rows: {}'.format( df1.shape[0] ) )

### 1.2.3 - DATA TYPES

In [ ]:
df1.dtypes

### 1.2.4 - CHECK NA VALUES

In [ ]:
df1.isna().sum()

#### 1.2.4.1 - HANDLING MISSING VALUES

In [ ]:
# To analyse the data, two new data set are being created:
df_missing = df1.loc[df1['customer_id'].isna(), :] # Contains the data with missing customer_id
df_not_missing = df1.loc[~df1['customer_id'].isna(),:] # Contain the data with customer_id

In [ ]:
df_not_missing.isna().sum()

In [ ]:
df_missing.isna().sum()

In [ ]:
# Identifying which are the invoices without customer id
missing_invoice = df_missing['invoice_no'].drop_duplicates().tolist()

# Locate the missing customer_id data by searching from invoice_no
df_aux = df_not_missing.loc[df_not_missing['invoice_no'].isin(missing_invoice)]

# Print result if any
df_aux.head()

In [ ]:
# The analysis will continue with the data set without NaN values
df1 = df_not_missing.copy()

**COMMENTS:**

Registers with missing description also has a missing customer_id.

Some actions could be taken in order to handle these issues:
An unique customer_id and invoice_no can have more than one register, as the granularity is defined by sotck_code, then:

1) Compare the invoice_no from the dataset containing missing values with the invoice_no from the complete dataset, and fill in the NA values where the entries match by invoice_no;
2) Add a random number to these customers, so the data would not be lost;
3) Drop NaN values;

The decision is to continue the project by doing the check described on (1) and if does not work, the NaN values will be removed, once the aim is to identify the best customers;

### 1.2.5 - CHANGE DTYPES

In [ ]:
df1.dtypes

In [ ]:
df1[ 'invoice_date'] = pd.to_datetime( df1['invoice_date'], format='%d-%b-%y') # changing the data on the column invoice data to match the correct data type

df1['customer_id'] = df1['customer_id'].astype( int ) # changing the data on the column customer id data to match the correct data type

df1['country'] = df1['country'].astype( str ) # Changing data type from object to string

### 1.2.6 - DESCRIPTIVE STATISTICS

In [ ]:
# For Numerical Data
df1.describe()

In [ ]:
# For Categorical Data
df1.describe(include='object')

**INSIGHTS**

*1. QUANTITY*

Approximately 75% of the 406,829 orders contain a maximum of 12 items. The mean and standard deviation appear to be impacted by high values. Upon reviewing the minimum and maximum values, two extremes are evident: the minimum is a negative number that exactly matches the maximum value, which suggests a return, cancellation, or another issue. This anomaly requires further investigation.

*2. UNIT PRICE*

The products appear to be relatively inexpensive, with an average price of 3.46. Additionally, 75% of the 406,829 entries have prices below 3.75. The minimum and maximum values are of particular interest—a minimum price of 0 likely indicates an error, and a maximum of approximately 38,000 suggests issues that should be examined in more detail.

*3. INVOICE NO*

There are 22,190 unique invoice numbers, indicating that the e-commerce platform processed 22,190 transactions during the period from 29/11/2016 to 08/07/2017. Additionally, invoice number 576339 contains 542 entries.

*4. STOCK CODE*

There are 3,684 unique products sold, with product 85123A leading as the top item by transaction volume.

*5. DESCRIPTION*

There are 3,896 unique descriptions compared to 3,684 unique stock codes, suggesting that some products may have multiple descriptions associated with the same stock code.

*6. COUNTRY*

The transactions span 37 different countries, with the majority originating from the UK.

### 1.3 - EDA

#### 1.3.1 - Univariate Analysis

##### QUANTITY

Analysing the **quantity** feature, it came to my attention that:

1) Some extremely values of quantity have an very similar data with a negative quantity and the invoice_no starts with "C". In my understanding, it means that some invoices have been canceled or returned.

I will:

Filter and remove the purchase register and the return register from the analysis.

In [ ]:
df1.nlargest(5, 'quantity')

In [ ]:
df1.nsmallest(5, 'quantity')

In [ ]:
sns.violinplot(x=df1['quantity'])

##### UNIT PRICE

There are some items with an extreme values as well as items with price equal to 0; These datas also have some odd stock code and quantities and they would be filtered on the step 2.

In [ ]:
df1.nlargest(5, 'unit_price')

In [ ]:
df1.nsmallest(5, 'unit_price')

In [ ]:
sns.violinplot(x=df1['unit_price'])

##### STOCK CODE

While analysing the data, it come to attention some stcock_code that does not reffers to products. A filter will be added, aiming to:

POST - Remove. It seems to be a delivery price. Need's to be confirmed with the company;

M    - Keep. It seems to be manual insertions and legit acquisition;

C2   - Remove. It seems to be a delivery price. Need's to be confirmed with the company;

D    - Remove. It is a discount applied on some acquisitions;

DOT  - Remove. It seems to be a delivery price. Need's to be confirmed with the company;

CRUK - Remove. It seems to be kind of comission.

PADS - Remove. Need's to be confirmed

BANK CHARGES - Remove. Bank fees

In [ ]:
df1["length_sc"] = df1["stock_code"].str.strip().str.len()

df1["length_sc"].value_counts(normalize=True)

In [ ]:
df1[df1["length_sc"] < 5]["stock_code"].value_counts(normalize=True)

In [ ]:
df1[df1["length_sc"] > 6]["stock_code"].value_counts(normalize=True)

Stock codes to be removed from the analysis
['POST','C2','DOT','PADS','BANK CHARGES'] 

##### COUNTRY

There are 345 registers where it is not possible to identify the country where the customer is from; Howevere these registers will be kept
once still is possible to identify the customer;

In [ ]:
df1['country'].value_counts(normalize=True)

In [ ]:
# How many registers does not specify the country?

dfc_aux = df1[['country', 'customer_id']].groupby('country').count().reset_index()
dfc_aux = dfc_aux.sort_values(by='customer_id', ascending=False).reset_index(drop=True)
dfc_aux[dfc_aux['country'].isin(['Unspecified', 'European Community'])]


##### QUESTIONS TO ANWSWER:

- Qual cliente comprou mais?
- Qual cliente devolveu mais?
- Qual produto foi o mais vendido?
- Qual produto foi o mais devolvido/cancelado?
- Qual mes vendeu mais?
- Qual mes vendeu menos?
- Qual dia da semana vendeu mais?
- Qual dia da semana vendeu menos?
- Qual pais concentra a maior parte do faturamento?

In [ ]:
df1.head()

#### 1.3.2 - Bivariate Analysis

# 2 - DATA FILTERING

On this project, the step **DATA FILTERING** will be done earlier as some features are been calculated from the dataset and are combined with that dataset later on, and it's our duty to make sure that we are keeping our data clean and without errors.

In [ ]:
df2 = df1.copy()

In [ ]:
# REMOVING REGISTERS WHERE THE PURCHASE HAVE BEEN RETURNED OR CANCELLED
# Classifying each invoice as cancelled (True) or not (False) 
df2['invoice_cancelled'] = df2['invoice_no'].str.startswith("C") & (df2['quantity']<0)

# Separting two datasets:
aux_cancelled = df2[df2['invoice_cancelled']] # invoices cancelled
aux_not_cancelled = df2[~df2['invoice_cancelled']] # invoices not cancelled

# Mergin the above two data sets
merged_df = aux_not_cancelled.merge(aux_cancelled, on=['stock_code','unit_price','customer_id'], how='left')

# Applying function to clean the new data set by removing columns etc.
df2 = drop_and_rename_duplicate_columns(merged_df)

# --- NUMERICAL ATTRIBUTES ---

# Filtering products where price is equal to 0
df2 = df2.loc[df2['unit_price'] > 0, :]

# --- CATEGORICAL ATTRIBUTES ---

# Filtering stock_codes that does not reffers to items
df2 = df2[~df2['stock_code'].isin( ['POST','C2','DOT','PADS','BANK CHARGES'] )]

# Description
df2 = df2.drop( columns='description', axis=1)

# Quantity
df2_returns = aux_cancelled.copy()
df2_purchases = aux_not_cancelled.copy()

# 3 - FEATURE ENGINEERING

In [ ]:
df3 = df2.copy()

In [ ]:
# data reference
df_ref = df2.drop( ['invoice_no','stock_code','quantity','invoice_date','unit_price','country'],axis=1).drop_duplicates(ignore_index=True)

## 3.1 - Gross Revenue

In [ ]:
# Calculus of the monetary value sold
df2.loc[:,'gross_revenue'] = df2.loc[:,'quantity'] * df2.loc[:,'unit_price']
df_sold = df2.loc[:, ['customer_id', 'gross_revenue']].groupby('customer_id').sum().reset_index()

df_ref = pd.merge( df_ref, df_sold, on='customer_id', how='left')

df_ref.isna().sum()

In [ ]:
df_ref.duplicated().sum()

## 3.2 - Recency - Day from last purchase

In [ ]:
# Recency
df_recency = df2.loc[:, ['customer_id', 'invoice_date']].groupby('customer_id').max().reset_index()
df_recency['recency_days'] = (df2['invoice_date'].max() - df_recency['invoice_date']).dt.days
df_recency = df_recency[['customer_id','recency_days']].copy()
df_ref = pd.merge( df_ref, df_recency, on='customer_id', how='left')

df_ref.isna().sum()

In [ ]:
df_ref.head()

## 3.3 - Qty of purchases

In [ ]:
df_freq = (df2.loc[:, ['customer_id', 'invoice_no']].drop_duplicates()
                                                    .groupby('customer_id')
                                                    .count()
                                                    .reset_index()
                                                    .rename( columns={'invoice_no':'qty_invoices'}) )
                
df_ref = pd.merge(df_ref, df_freq, on='customer_id', how='left')

df_ref.isna().sum()

In [ ]:
df_ref.head()

## 3.4 Qty of products purchased

In [ ]:
# Number of products
df_freqp = ( df2.loc[:,['customer_id', 'quantity']].groupby('customer_id')
                                                   .sum().reset_index()
                                                   .rename( columns={'quantity':'qty_prod_purchased'}) )

df_ref = pd.merge(df_ref, df_freqp, on='customer_id', how='left')
df_ref.isna().sum()

In [ ]:
df_ref.head()

## 3.5 - Range of Products per Customer

In [ ]:
df_prod = ( df2.loc[:,['customer_id', 'stock_code']].groupby('customer_id')
                                                    .count()
                                                    .reset_index()
                                                    .rename( columns={'stock_code':'range_of_products'}) )
            
df_ref = pd.merge(df_ref, df_prod, on='customer_id', how='left')
df_ref.isna().sum()

## 3.6 - Average Ticket Value

In [ ]:
# Avg Ticket
df_avg_ticket = ( df2.loc[:, ['customer_id','gross_revenue']].groupby('customer_id')
                                                             .mean()
                                                             .reset_index()
                                                             .rename( columns={'gross_revenue':'avg_ticket'}) )

df_ref = pd.merge( df_ref, df_avg_ticket, on='customer_id', how='left')
df_ref.isna().sum()

## 3.7 - Average Recency Days

In [ ]:
# df_aux = ( df2_purchases[['customer_id','invoice_date']].drop_duplicates()
#                                                       .sort_values( ['customer_id', 'invoice_date'], ascending=[False, False] ) ) 
    
# df_aux['next_customer_id'] = df_aux['customer_id'].shift() # next customer
# df_aux['previous_date'] = df_aux['invoice_date'].shift() # next invoice date
# df_aux['avg_recency_days'] = df_aux.apply( lambda x: x['invoice_date'] - x['previous_date'] if x['customer_id'] == x['next_customer_id'] else np.nan, axis=1 ).dt.days

# df_recency = df_recency[['customer_id','recency_days']].copy() # calculating the recency day

# df_aux = df_aux.drop(['invoice_date','next_customer_id','previous_date'], axis=1).dropna()# leaving only the interested variable and the customer_id into the aux dataframe

# # calculating the average for customer
# df_avg_recency_days = df_aux.groupby('customer_id').mean().reset_index()

# # meging the result with the original df

# df_ref= pd.merge(df_ref, df_avg_recency_days, on='customer_id', how='left')

# df_ref.isna().sum()

## 3.8 - Frequency of Purchases

In [ ]:
df2_max = df2[['customer_id','invoice_date']].drop_duplicates().groupby('customer_id').max().reset_index() # finding the last date of purchase per customer
df2_min = df2[['customer_id','invoice_date']].drop_duplicates().groupby('customer_id').min().reset_index() # finding the last date of purchase per customer

df_aux = ( df2[['customer_id','invoice_no','invoice_date']].drop_duplicates()
                                                           .groupby('customer_id')
                                                           .agg( max_ =('invoice_date', 'max'),
                                                                 min_ =('invoice_date', 'min'),
                                                                 days_=('invoice_date', lambda x:( (x.max() - x.min() ).days)+1),
                                                                 buy_ =('invoice_no', 'count'))).reset_index()

# Frequency
df_aux['frequency'] = df_aux[['buy_', 'days_']].apply( lambda x: x['buy_'] / x['days_'] if x['days_'] != 0 else 0, axis=1 )

# Merge
df_ref = pd.merge(df_ref, df_aux[['customer_id','frequency']], on='customer_id', how='left')

df_ref.isna().sum()

## 3.9 - Returns

In [ ]:
# Number of returns
df_returns = df2_returns[['customer_id','quantity']].groupby('customer_id').sum().reset_index().rename( columns={'quantity':'qty_returns'})
df_returns['qty_returns'] = df_returns['qty_returns'] * -1

df_ref = pd.merge( df_ref, df_returns, on='customer_id', how='left')
df_ref.loc[df_ref['qty_returns'].isna(), 'qty_returns'] = 0

df_ref.isna().sum()

In [ ]:
df_ref.head()

## 3.10 - Qty avg of producst per customer

In [ ]:
df_ref['avg_qty_products_purchased'] = df_ref['qty_prod_purchased'] / df_ref['qty_invoices']
df_ref.isna().sum()

In [ ]:
df_ref.head()

In [ ]:
df_ref.duplicated().sum()

## 3.11 - Week day most frequent per customer

The day of the week with Monday=0, Sunday=6.

In [ ]:
# Retrieving the week day for the specific date
df2['week_day']= df2['invoice_date'].dt.dayofweek

# Creating the dataframe with day_week per invoice_no
aux_02 = df2[['invoice_no','customer_id','week_day']].drop_duplicates(ignore_index=True)

# Calculus of week day most frequent
aux_03 = aux_02[['customer_id', 'week_day']].groupby('customer_id').apply(lambda x: x.mode().iloc[0]).reset_index(drop=True)

# Adding the new feature into the data set
df_ref = pd.merge( df_ref, aux_03, on='customer_id', how='left')

df_ref.isna().sum()

In [ ]:
df_ref.head()

## 3.12 - month most frequent per customer

In [ ]:
# Retrieving the week day for the specific date
df2['month']= df2['invoice_date'].dt.month

# Creating the dataframe with day_week per invoice_no
aux_312 = df2[['invoice_no','customer_id','month']].drop_duplicates(ignore_index=True)

# Calculus of week day most frequent
aux_3123 = aux_312[['customer_id', 'month']].groupby('customer_id').apply(lambda x: x.mode().iloc[0]).reset_index(drop=True)

# Adding the new feature into the data set
df_ref = pd.merge( df_ref, aux_3123, on='customer_id', how='left')

df_ref.isna().sum()

# <font color='black'> 4 - EDA </font>

In [ ]:
df4 = df_ref.copy()

In [ ]:
# df4 = df_ref.dropna()
# df4.isna().sum()

## 4.1 - Univariate Analysis

Metrics to be checked out:

1. Cluster coesos - separados
2. Metrics:
    - Min, Max, Range (dispersion)
    - Mean and Median
    - Standard Deviation and Variance
    - CV (Coefficient of Variation)
    - Distribuition

In [ ]:
prof = ProfileReport(df4)
prof.to_file('data_descriptive.html') 

## 4.2 - Bivariate Analysis

In [ ]:
# cols = ['customer_id']
# df42 = df4.drop( cols, axis=1 )

In [ ]:
# plt.figure( figsize=(12, 15) )
# sns.pairplot( df4 )

### Quais sao os 20 customer_id que gastaram mais dinheiro?

In [ ]:
# Agrupar por 'customer_id' e calcular a soma de 'gross_revenue'
aux = df4.groupby('customer_id')['gross_revenue'].sum().reset_index()

# Ordenar por 'gross_revenue' de forma decrescente e pegar o top 20
aux = aux.sort_values(by='gross_revenue', ascending=False).head(20)

# Definindo a figura e o eixo para o gráfico de barras
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=aux, x='customer_id', y='gross_revenue', ax=ax, order=aux['customer_id'])

# Adiciona rótulos a cada barra
for p in ax.patches:
    ax.annotate(format(p.get_height(), '.0f'), 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', xytext=(0, 10), 
                textcoords='offset points')

# DESIGN
# Oculta os valores do eixo y
ax.set_yticks([])  # Remove ticks do eixo y

# Rotaciona os rótulos do eixo x em 45 graus
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')


# Título
plt.text(10, 300000, s='Top 20 Customers by Highest Expenses', 
         fontsize=15, color='#8C908F', weight='bold')

plt.tight_layout()
plt.show()

### Quais sao os 20 customer_id que realizaram mais compras?

In [ ]:
# Agrupar por 'customer_id' e calcular a soma de 'invoice_no'
aux = df4[['customer_id','qty_invoices']].sort_values(by='qty_invoices', ascending=False)

# Ordenar por 'gross_revenue' de forma decrescente e pegar o top 20
aux = aux.sort_values(by='qty_invoices', ascending=False).head(20)

# Definindo a figura e o eixo para o gráfico de barras
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=aux, x='customer_id', y='qty_invoices', ax=ax, order=aux['qty_invoices'])

# Adiciona rótulos a cada barra
for p in ax.patches:
    ax.annotate(format(p.get_height(), '.0f'), 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', xytext=(0, 10), 
                textcoords='offset points')

# DESIGN
# Oculta os valores do eixo y
ax.set_yticks([])  # Remove ticks do eixo y

# Rotaciona os rótulos do eixo x em 45 graus
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')


# Título
plt.text(10, 300000, s='Top 20 Customers by Qty of Invoices', 
         fontsize=15, color='#8C908F', weight='bold')

plt.tight_layout()
plt.show()

### Quais os 20 customer_id que retornaram ou cancelaram mais pedidos?

In [ ]:
aux = df2_returns[['invoice_no','customer_id']].drop_duplicates()
aux01 = aux[['customer_id','invoice_no']].groupby('customer_id').count().reset_index()
aux01.sort_values(by='invoice_no', ascending=False).head(20)

### Qual o produto menos vendido?

In [ ]:
aux = df2[['stock_code','quantity']].groupby('stock_code').sum().reset_index()
aux.sort_values(by='quantity', ascending=True).head(20)

### Qual o produto mais vendido?

In [ ]:
aux = df2[['stock_code','quantity']].groupby('stock_code').sum().reset_index()
aux.sort_values(by='quantity', ascending=False).head(20)

### Qual pais emitiu maior numero de invoices?

In [ ]:
import plotly.graph_objects as go
from plotly.offline import iplot

In [ ]:
temp = df2[['customer_id', 'invoice_no', 'country']].groupby(['customer_id', 'invoice_no', 'country']).count().reset_index()
countries = temp['country'].value_counts()

In [ ]:
# Dados para o gráfico
data = dict(
    type='choropleth',
    locations=countries.index,
    locationmode='country names',
    z=countries,
    text=countries.index,
    colorbar={'title': 'Order nb.'},
    colorscale=[
        [0, 'rgb(224,255,255)'],
        [0.01, 'rgb(166,206,227)'],
        [0.02, 'rgb(31,120,180)'],
        [0.03, 'rgb(178,223,138)'],
        [0.05, 'rgb(51,160,44)'],
        [0.10, 'rgb(251,154,153)'],
        [0.20, 'rgb(255,255,0)'],
        [1, 'rgb(227,26,28)']
    ],
    reversescale=False
)

# Layout do gráfico com título centralizado
layout = dict(
    title=dict(
        text='Number of orders per country',
        x=0.5,  # Centraliza o título
        xanchor='center'
    ),
    geo=dict(
        showframe=True,
        projection={'type': 'mercator'}
    ),
    width=1000,  # Largura do gráfico
    height=700   # Altura do gráfico
)

# Gerar e exibir o gráfico
choromap = go.Figure(data=[data], layout=layout)
iplot(choromap, validate=False)

## 4.3 - Space of Study

In [ ]:
# Original Dataset
df43 = df4.drop( columns=['customer_id'], axis=1 ).copy()

# # Selected Dataset
# cols_selected = ['customer_id', 'gross_revenue', 'recency_days', 'qty_of_products', 'frequency' , 'qty_returns']
# df43 = df4[ cols_selected ]

In [ ]:
# # Standardization
# mms = pp.MinMaxScaler()

# # Transformer
# df43['gross_revenue']     = mms.fit_transform( df43[['gross_revenue']])
# df43['recency_days']      = mms.fit_transform( df43[['recency_days']])
# #df43['qty_invoices']      = mms.fit_transform( df43[['qty_invoices']])
# df43['qty_of_products']   = mms.fit_transform( df43[['qty_of_products']])
# #df43['range_of_products'] = mms.fit_transform( df43[['range_of_products']])
# #df43['avg_ticket']        = mms.fit_transform( df43[['avg_ticket']])
# #df43['avg_recency_days']  = mms.fit_transform( df43[['avg_recency_days']])
# df43['frequency']         = mms.fit_transform( df43[['frequency']])
# df43['qty_returns']       = mms.fit_transform( df43[['qty_returns']])
# #df43['n_purchase']        = mms.fit_transform( df43[['n_purchase']])
# #df43['n_products']        = mms.fit_transform( df43[['n_products']])
# #df43['avg_basket_size']   = mms.fit_transform( df43[['avg_basket_size']])

# X = df43.copy()

In [ ]:
# Standardization
mms = pp.MinMaxScaler()

# Transformer
df43['gross_revenue']              = mms.fit_transform( df43[['gross_revenue']])
df43['recency_days']               = mms.fit_transform( df43[['recency_days']])
df43['qty_invoices']               = mms.fit_transform( df43[['qty_invoices']])
df43['qty_prod_purchased']         = mms.fit_transform( df43[['qty_prod_purchased']])
df43['range_of_products']          = mms.fit_transform( df43[['range_of_products']])
df43['avg_ticket']                 = mms.fit_transform( df43[['avg_ticket']])
df43['frequency']                  = mms.fit_transform( df43[['frequency']])
#df43['qty_returns']                = mms.fit_transform( df43[['qty_returns']])
df43['avg_qty_products_purchased'] = mms.fit_transform( df43[['avg_qty_products_purchased']])
df43['week_day']                   = mms.fit_transform( df43[['week_day']])

X = df43.copy()

In [ ]:
X.head()

In [ ]:
X.shape

### 4.3.1 - PCA

In [ ]:
fig, axs = plt.subplots(figsize = (8, 4), dpi=150)
pca = dd.PCA ( n_components = X.shape[1] )

principal_components = pca.fit_transform( X )

# plot explained variables
features = range( pca.n_components_ )

plt.bar( features, pca.explained_variance_ratio_, color='black' )

# pca component

df_pca = pd.DataFrame( principal_components )

In [ ]:
sns.scatterplot( x=0, y=1, data=df_pca)

### 4.3.2 - UMAP

In [ ]:
# UMAP 
reducer = umap.UMAP( random_state=42 )
embedding = reducer.fit_transform( X )

# Embedding 
df_pca['embedding_x'] = embedding[:, 0]
df_pca['embedding_y'] = embedding[:, 1]

# Plot UMAP
sns.scatterplot( 
                 x='embedding_x',
                 y='embedding_y',
                 data=df_pca)
                                

### 4.3.3 - TSNE

In [ ]:
# UMAP 
reducer = TSNE( n_components=2, n_jobs=-1, random_state=42 )
embedding = reducer.fit_transform( X )

# Embedding 
df_pca['embedding_x'] = embedding[:, 0]
df_pca['embedding_y'] = embedding[:, 1]

# Plot UMAP
sns.scatterplot( 
                 x='embedding_x',
                 y='embedding_y',
                 data=df_pca)
                                

### 4.3.4 - Tree-Based Embedding

In [ ]:
# Training dataset
X = df43.drop( columns=['gross_revenue'], axis=1 ).copy()
y = df43['gross_revenue']

# model definition
rf_model = en.RandomForestRegressor( n_estimators=100, random_state=42 )

# model trainig
rf_model.fit(X, y)

In [ ]:
df_leaf = pd.DataFrame( rf_model.apply( X ) ) # here i am selecting the datas from the leaf of tree

In [ ]:
# UMAP 
reducer = umap.UMAP( random_state = 42 )
embedding = reducer.fit_transform( df_leaf )

# Embedding 
df_tree = pd.DataFrame()
df_tree['embedding_x'] = embedding[:, 0]
df_tree['embedding_y'] = embedding[:, 1]

# Plot UMAP
sns.scatterplot( 
                 x='embedding_x',
                 y='embedding_y',
                 data=df_tree)

# 5 - DATA PREPARATION

In [ ]:
#df5 = df4.copy()
df5 = df_tree.copy()

In [ ]:
df5.head()

#### Standardization

In [ ]:
#mm = pp.MinMaxScaler()
#ss = pp.StandardScaler()
#rs = pp.RobustScaler()

#df5['gross_revenue']     = mm.fit_transform( df5[['gross_revenue']])

#df5['recency_days']      = mm.fit_transform( df5[['recency_days']])

#df5['qty_invoices']      = mm.fit_transform( df5[['qty_invoices']])

#df5['qty_of_products']   = mm.fit_transform( df5[['qty_of_products']])

#df5['range_of_products'] = mm.fit_transform( df5[['range_of_products']])

#df5['avg_ticket']        = mm.fit_transform( df5[['avg_ticket']])

#df5['avg_recency_days']  = mm.fit_transform( df5[['avg_recency_days']])

#df5['frequency']         = mm.fit_transform( df5[['frequency']])

#df5['qty_returns']       = mm.fit_transform( df5[['qty_returns']])

#df5['n_purchase']        = mm.fit_transform( df5[['n_purchase']])

#df5['n_products']        = mm.fit_transform( df5[['n_products']])

#df5['avg_basket_size']   = mm.fit_transform( df5[['avg_basket_size']])

#variable = 'qty_of_products'

In [ ]:
# Dados AS IS
#print('Min:{} - Max:{}'.format( df5_aux[variable].min(), df5_aux[variable].max() ) )
#sns.displot( df5_aux[variable]);

In [ ]:
# Dados Normalizados e Reescalados
#print('Min:{} - Max:{}'.format( df5[variable].min(), df5[variable].max() ) )
#sns.displot( df5[variable]);

In [ ]:
# BoxPlot
#sns.boxplot(df5_aux[variable]);

# 6 - FEATURE SELECTION

In [ ]:
cols_selected = ['customer_id', 'gross_revenue', 'recency_days', 'qty_of_products', 'frequency' , 'qty_returns']

In [ ]:
#df6 = df5[cols_selected].copy()

In [ ]:
df6 = df_tree.copy()

# 7 - FINE TUNING

In [ ]:
X = df_tree.copy()

In [ ]:
cluster = np.arange(2, 25, 1)

## 7.1 - K-Means

In [ ]:
kmeans_list = []
for k in cluster:
    # Model Definition
    kmeans_model = cc.KMeans( n_clusters=k )

    # Model Training
    kmeans_model.fit( X )
    # Model Predict
    labels = kmeans_model.predict( X )

    # Model Performance
    sil = mt.silhouette_score( X, labels, metric='euclidean')
    kmeans_list.append( sil )

In [ ]:
plt.figure( figsize=(8, 5) )
plt.plot( cluster, kmeans_list, linestyle='--', marker='o', color='b')
plt.xlabel( 'K' );
plt.ylabel( 'Silhouette Score');
plt.title('KMeans: Silhouette Score x K');

## 7.2 - Gaussiann Mixture Model

AIC - Ajuste dos dados
BIC - Ajuste dos parametros

In [ ]:
gmm_list = [] 
for k in cluster:
    # Model Definition
    gmm_model = mx.GaussianMixture( n_components=k )
    
    # Model Training
    gmm_model.fit( X )

    # Model Predict
    labels = gmm_model.predict( X )

    # Model Performance
    gmm_sil = mt.silhouette_score( X, labels, metric='euclidean')
    gmm_list.append( gmm_sil )

In [ ]:
plt.figure( figsize=(8, 5) )
plt.plot( cluster, gmm_list, linestyle='--', marker='o', color='b')
plt.xlabel( 'K' );
plt.ylabel( 'Silhouette Score');
plt.title('GMM: Silhouette Score x K')

## 7.3 - Hierarchical Clustering

In [ ]:
from scipy.cluster import hierarchy as hc

In [ ]:
# Model Definition and Training
hc_model = hc.linkage( X, 'ward')

In [ ]:
hc.dendrogram(
              hc_model,
              leaf_rotation=90,
              leaf_font_size=8
)

plt.plot()

In [ ]:
hc.dendrogram(
            hc_model,
            truncate_mode='lastp',
            p=12,
            leaf_rotation=90,
            leaf_font_size=8,
            show_contracted=True
)
plt.show()

### 7.3.1 - Hierarchical Cluster Silhouette

In [ ]:
hc_list = []
for k in cluster:
    # Model Definition and Training
    hc_model = hc.linkage( X, 'ward' )

    # Model Predict
    labels = hc.fcluster( hc_model, k, criterion='maxclust' )

    # metrics
    sil = mt.silhouette_score( X, labels, metric='euclidean' )
    hc_list.append( sil )

In [ ]:
plt.figure( figsize=(8, 5) )
plt.plot( cluster, hc_list, linestyle='--', marker='o', color='b')
plt.xlabel( 'K' );
plt.ylabel( 'Silhouette Score');
plt.title('HC: Silhouette Score x K')

In [ ]:
hc_list

## 7.4 - DBSCAN

In [ ]:
# eps=0.01
# min_samples=20

# # Model Definition
# dbscan_model = cc.DBSCAN( eps=eps, min_samples=min_samples )

# # Model Training and Predict
# labels = dbscan_model.fit_predict( X )

# db_sil = mt.silhouette_score( X, labels, metric='euclidean' )
# print( 'Silhouette Score: {}'.format( sil ) )
# print( 'Number of Cluster: {}'.format( len( unique( labels) ) ) )
# print( unique( labels ))

In [ ]:
# from sklearn.neighbors import NearestNeighbors

In [ ]:
# neighbors = NearestNeighbors( n_neighbors = min_samples).fit( X )
# distances, indexes = neighbors.kneighbors( X )

In [ ]:
# distances = np.sort( distances, axis=0 )
# distances = distances[:, 1]

In [ ]:
# plt.figure( figsize=(8, 5) )
# plt.plot( distances )

## 7.5 - Silhouette Analysis

In [ ]:
fig, ax = plt.subplots( 13, 2 )
fig.set_size_inches( 25, 20 )

for k in cluster:
    q, mod = divmod( k, 2 )
    
    ax[q-1, mod].set_xlim( [-0.1, 1] )
    ax[q-1, mod].set_ylim( [0, len( X ) + ( k+1 )*10] )
    
    # Model Definition and Training
    hc_model = hc.linkage( X, 'ward')

    # Model Predict
    labels = hc.fcluster( hc_model, k, criterion='maxclust')
    
    # Performance
    ss = mt.silhouette_score( X, labels, metric='euclidean')
    print( ' For K = {}. Silhouette Score: {}'.format( k, ss ) )

    
    samples_silhouette_values = mt.silhouette_samples( X, labels )

    y_lower = 10
    for i in range( k ):
        # Select Clusters
        ith_samples = samples_silhouette_values[ labels == i ]

        # Size Clusters
        size_cluster_i = ith_samples.shape[0]
        
        # Sort values
        ith_samples.sort()    

        # Limits
        y_upper = y_lower + size_cluster_i

        ax[q-1, mod].fill_betweenx( np.arange( y_lower, y_upper ), 0, ith_samples )
        
        y_lower = y_upper + 10 
     
        cmap = cm.get_cmap( 'Spectral')
        color = cmap( i/k )
    
    
    
    
ax[q-1, mod].set_yticks([])
ax[q-1, mod].set_xticks([-0.1, 0.2, 0.4, 0.6, 0.8, 1])

## 7.6 - Results

In [ ]:
# Model Name / K=2 / K=3 / K=4 / K=5 / ...
# KMeans       SS    SS    SS    SS
# GMM          SS    SS    SS    SS
# HC           SS    SS    SS    SS
# DBSCAN       SS    SS    SS    SS

In [ ]:
df_results = pd.DataFrame(
                          {'KMeans': kmeans_list,
                           'GMM': gmm_list,
                           'HC':hc_list}
).T

df_results.columns = cluster

In [ ]:
df_results.style.highlight_max( color='lightgreen', axis=1 ) #identifying the highest value for each ml model

# 8 - MACHINE LEARNING TRAINING

## 8.1 - K-Means

In [ ]:
# # Model Definition
# k = 14
# kmeans = cc.KMeans( init='random', n_clusters=k, n_init=300, random_state=42 )

# # Model Trainin
# kmeans.fit( X )

# # Model Prediction
# labels = kmeans.labels_

## 8.2 - GMM

In [ ]:
k = 20
# Model Definition
gmm_model = mx.GaussianMixture( n_components=k, n_init=300, random_state=42  )
    
# Model Training
gmm_model.fit( X )

# Model Predict
labels = gmm_model.predict( X )

## 8.3 - HC

In [ ]:
# k=18
# # Model Definition and Training
# hc_model = hc.linkage( X, 'ward' )

# # Model Predict
# labels = hc.fcluster( hc_model, k, criterion='maxclust' )

## 8.2 - Cluster Validation

In [ ]:
# WSS
print('WSS value: {}'. format( kmeans.inertia_) )

# Silhouette Score
print( 'SS Score: {}'.format( mt.silhouette_score( X, labels, metric='euclidean' ) ) )

# 9 - CLUSTER ANALYSIS

In [ ]:
df9 = X.copy()
df9['cluster'] = labels

## 9.1 - Vizualization Inspection


In [ ]:
# Definindo a figura e os eixos para o gráfico de dispersão
fig, ax = plt.subplots(figsize=(15, 7))

# Gráfico de dispersão com legenda de cores para clusters
sns.scatterplot(x='embedding_x', y='embedding_y', hue='cluster', data=df9, palette=sns.color_palette("hls", 18), ax=ax)

# Título
plt.text(x=2, y=25, s='Cluster formation', fontsize=25, color='#8C908F', weight='bold')

# Movendo a legenda para fora do gráfico
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="Cluster")  # Ajuste de posição da legenda

fig.tight_layout()
plt.show()

## 9.2 - 2D Plot

In [ ]:
# df_viz = df9.drop( columns='customer_id', axis=1)
# sns.pairplot( df_viz, hue='cluster')

## 9.3 - UMAP

Machine Learning - Manifold

Aprendizado por topologia: PCA - Metodo baseado em matriz ou espaco entre distancias. Temos 9 condicoes, cumprir 9 colorarios para ter uma garantia de espaco. Espaco de Hilbert, etc.AffinityPropagation

UMAP, T-SNE ( 2009 )- Abordagem por topologia ( Manifold ). Topologia sao grafos em alta dimensionalidade

In [ ]:
# # UMAP 
# reducer = umap.UMAP(n_neighbors=90, random_state=42)
# embedding = reducer.fit_transform( X )

# # Embedding 
# df_viz['embedding_x'] = embedding[:, 0]
# df_viz['embedding_y'] = embedding[:, 1]

# # Plot UMAP
# sns.scatterplot( 
#                  x='embedding_x',
#                  y='embedding_y', 
#                  hue='cluster',
#                  palette=sns.color_palette('hls',
#                                            n_colors=len(df_viz['cluster'].unique())),
#                  data=df_viz)
                                           


## 9.4 - Cluster Profile

In [ ]:
df94 = df4.copy()
df94['cluster'] = labels
df94.head()

In [ ]:
df94[['gross_revenue','cluster']].groupby('cluster').sum().reset_index()

In [ ]:
# Number of Customer
df_cluster = df94[['customer_id', 'cluster']].groupby('cluster').count().reset_index()
df_cluster['perc_customer'] = 100*( df_cluster['customer_id'] / df_cluster['customer_id'].sum() )

# Avg Gross Revenue
df_avg_gross_revenue = df94[['gross_revenue', 'cluster']].groupby('cluster').mean().reset_index()
df_cluster = pd.merge( df_cluster, df_avg_gross_revenue, how='inner', on='cluster')

# Avg Recency Days
df_avg_recency_days = df94[['recency_days', 'cluster']].groupby('cluster').mean().reset_index()
df_cluster = pd.merge( df_cluster, df_avg_recency_days, how='inner', on='cluster')

# Avg Qty Products
df_qty_product = df94[['qty_prod_purchased', 'cluster']].groupby('cluster').mean().reset_index()
df_cluster = pd.merge( df_cluster, df_qty_product, how='inner', on='cluster')

# Avg Frequency
df_freq = df94[['frequency', 'cluster']].groupby('cluster').mean().reset_index()
df_cluster = pd.merge( df_cluster, df_freq, how='inner', on='cluster')

# Avg Returns
#df_ret = df94[['qty_returns', 'cluster']].groupby('cluster').mean().reset_index()
#df_cluster = pd.merge( df_cluster, df_ret, how='inner', on='cluster')

df_cluster.sort_values(by='gross_revenue', ascending=False)

In [ ]:
# 18 - Insiders:


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
19

# 10 - EXPLORATORY DATA ANALYSIS

## 10.1 - Mind Map of Hypothesis

1. Phenomen
2. Entity
3. caracteristics of Entity

In [ ]:
# URL da imagem
url_imagem = '../reports/figures/mindmap.png'  # Substitua pela URL da sua imagem

# Exibe a imagem no notebook
display(Image(url=url_imagem))

## 10.2 - Business Hypothesis

In [ ]:
Cluster | Comparation Basys | Variable to test

## 10.3 - Hypothesis Priority

# Purchases

1. Customers in the insider cluster make their payments using credit card in 80% of their purchases.
2. **Customers in the insider cluster has an avg ticket 10% higher than the cluster A.**
3. **Customers in the insider cluster has a basket size containing more than 5 items.**
4. **Customers in the insider cluster has a purchase volume higher than 10% of the total.**
5. **Customers in the insider clsuter has less returns than the average of the database.**

In [ ]:
# Customers
1. 60% of the customers in the insiders cluster are single. 
2. 10% of the customers in the insiders cluster are from 24 to 35 years old.
3. 40% of delivery locations are within a 50km radius.
4. 5% of the customers in the insiders cluster receive more than 100.000 dolars annualy.
5. 90% of the customers in the insiders cluster hold a BSc.

In [ ]:
# Products
1. 30% of all products are large
2. The median prices of products purchased by customers in the insider cluster are 10% higher than the median prices of all products.
3. The percentile prices of the products bought from the insider cluster.
4. The average weight of the products bought from customers in the insider cluster is higher than the average weight of the products bought from customers in the other clusters.
5. The average age of products purchased by customers in the insider cluster is less than 15 days.

In [ ]:
# Business Question
1. Who are the eligible individuals to participate in the program?
    - Revenue:
        - High average ticket value
        - High customer lifetime value (LTV)
        - Low recency
        - Low churn probability
        - High LTV prediction
        - High purchase propensity
        - Cost:

        - Low return rate
        - Purchase Experience:

        - High average ratings  

2. How many customers will be part of the group?
        - Total number of customers
        - % of the insiders group
    
3.  What are the main characteristics of these customers?
    - Customer characteristics:
        - Age
        - Location
       
    - Consumption characteristics:
        - Attributes of clustering

4. What percentage of the revenue comes from the selected group?
    - Total annual revenue
    - Revenue from the Insiders group

5. What is the revenue expectation for this group in the upcoming months?
    - LTV (Customer Lifetime Value) of the Insiders group
    - Cohort analysis
    
6. What are the eligibility criteria for someone to join the group?
    - Define the frequency (1 month, 2 months, etc.).
    - The person needs to be similar to another person in the group.
    - What are the criteria for someone to be removed from the group?

7. Define the frequency (1 month, 2 months, etc.).
    - The person needs to be dissimilar to another person in the group.
    - How can we ensure that the group performs better than the rest of the customer base?

8. A/B testing.
    - Bayesian A/B testing.
    - Hypothesis testing.
    - What actions can the marketing team take to increase revenue?

9. Discounts.
    - Purchase preferences.
    - Company visits.

## 10.4 - Hypothesis Validation

In [ ]:
df10 = df94.copy()
df10.head()

In [ ]:
# Purchases

1. Customers in the insider cluster make their payments using credit card in 80% of their purchases.
2. Customers in the insider cluster has an avg ticket 10% higher than the cluster A.
3. **Customers in the insider cluster has a basket size containing more than 5 items.**
4. **Customers in the insider cluster have a purchase volume (product) higher than 10% of the total.**
5. **Customers in the insider cluster has a purchase  volume (revenue) higher than 10% of the total.**
6. **Customers in the insider clsuter has less returns than the average of the database.**

# Customers
1. 60% of the customers in the insiders cluster are single. 
2. 10% of the customers in the insiders cluster are from 24 to 35 years old.
3. 40% of delivery locations are within a 50km radius.
4. 5% of the customers in the insiders cluster receive more than 100.000 dolars annualy.
5. 90% of the customers in the insiders cluster hold a BSc.

# Products
1. 30% of all products are large
2. The median prices of products purchased by customers in the insider cluster are 10% higher than the median prices of all products.
3. The percentile prices of the products bought from the insider cluster.
4. The average weight of the products bought from customers in the insider cluster is higher than the average weight of the products bought from customers in the other clusters.
5. The average age of products purchased by customers in the insider cluster is less than 15 days.


### 4. **Customers in the insider cluster have a purchase volume (product) higher than 10% of the total.** 
**Hiphotesis is True.** The Insider Cluster has a volume of purchases around 57%

In [ ]:
df_sales_insider = df10.loc[df10['cluster'] == 18, 'qty_prod_purchased'].sum()
df_sales_total = df10.loc[:,'qty_prod_purchased'].sum()

print('% Sales Insiders: {:.2f}%'.format( 100*df_sales_insider / df_sales_total ) )

### 5. **Customers in the insider cluster has a purchase  volume (revenue) higher than 10% of the total.**
**Hipothesis is True.** The Insider Cluster has a GMV volume of 55%.

In [ ]:
df_gmv_insider = df10.loc[df10['cluster'] == 18, 'gross_revenue'].sum()
df_gmv_total = df10.loc[:,'gross_revenue'].sum()

print('% GMV Insiders: {:.2f}%'.format( 100*df_gmv_insider / df_gmv_total ) )

### 6. **Customers in the insider clsuter has less returns than the average of the database.**
**Hipothesis is False.** The Insider Cluster has return avg higher than the general average.

In [ ]:
df_avg_return_insider = df10.loc[df10['cluster'] == 18, 'qty_returns'].mean()
df_avg_return_all = df10.loc[:,'qty_returns'].mean()

print('Avg Return Insiders: {} vs Avg Return Al: {}'.format( np.round( df_avg_return_insider, 0),
                                                             np.round( df_avg_return_all, 0 ) ) )

### The median revenue of the customers in the insider cluster are 10% higher than the median prices of all.
**Hipothesis is True.** The revenue median of the customers in the insider cluster is around 500% higher.

In [ ]:
df_median_gmv_insider = df10.loc[df10['cluster'] == 18, 'gross_revenue'].median()

df_median_gmv_total = df10.loc[:, 'gross_revenue'].median()

gmv_dif = ( df_median_gmv_insider - df_median_gmv_total )/ df_median_gmv_total

print('Median Diff: {:.2f}%'.format( 100 * gmv_dif ) )

### The percentile prices of the products bought from the insider cluster.


In [ ]:
np.percentile( df10.loc[df10['cluster'] == 18, 'gross_revenue'], q=0.1)

In [ ]:
np.percentile( df10.loc[df10['cluster'] == 18, 'gross_revenue'], q=0.9)

In [ ]:
sns.violinplot( x = df10.loc[df10['cluster'] == 18, 'gross_revenue']);

In [ ]:
df_aux = df10.loc[(df10['cluster'] == 1) & (df10['gross_revenue'] < 10000), 'gross_revenue']
sns.violinplot( x = df_aux )

## 10.5 - Business Questions

In [ ]:
1. Whom are the customers elegible to join the insiders program?

In [ ]:
df10.loc[df10['cluster'] == 18, 'customer_id'].head()

In [ ]:
2. How many customers will be part of the group?

In [ ]:
df10.loc[df10['cluster'] == 18, 'customer_id'].size

In [ ]:
3. What are the main characteristics of these customers?

In [ ]:
- No. customers: 495 customers (8.7%)
- Recency: 131 days
- Qty. Products: 3440
- Avg Revenue: $6171,70

In [ ]:
4. What percentage of the revenue comes from the selected group?

In [ ]:
df_insiders_gmv = df10.loc[df10['cluster'] == 18, 'gross_revenue'].sum()
df_all_gmv = df10.loc[:, 'gross_revenue'].sum()

print( '% GMV from Insiders: {}'.format( df_insiders_gmv / df_all_gmv ))

In [ ]:
5. What is the revenue expectation for this group in the upcoming months?

In [ ]:
6. What are the conditions for someone to be eligible for the group?

In [ ]:
7. What are the conditions for someone to be removed from the group?

In [ ]:
8. How do we know that the group is better than the rest of the base?

In [ ]:
9. What actions can the marketing team take to increase revenue?

# 11 - DEPLOY